# FloodNet — Training Notebook
**Model:** U-Net + ResNet50 encoder (pretrained ImageNet)  
**Device:** Colab T4 GPU  
**Target:** ~20 epochs in ~40 minutes  
**Key metric:** mIoU, especially IoU for class 1 (Bldg Flooded)

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────────────────────
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
# ── Cell 2: Mount Drive + paths ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
BASE       = "/content/drive/MyDrive/floodnet/FloodNet-Supervised_v1.0"
TRAIN_IMG  = os.path.join(BASE, "train/train-org-img")
TRAIN_MASK = os.path.join(BASE, "train/train-label-img")
VAL_IMG    = os.path.join(BASE, "val/val-org-img")
VAL_MASK   = os.path.join(BASE, "val/val-label-img")
TEST_IMG   = os.path.join(BASE, "test/test-org-img")
TEST_MASK  = os.path.join(BASE, "test/test-label-img")
CKPT_DIR   = "/content/drive/MyDrive/floodnet/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"Train images: {len(os.listdir(TRAIN_IMG))}")
print(f"Val images  : {len(os.listdir(VAL_IMG))}")
print("Drive mounted OK")

In [ ]:
# ── Cell 3: Imports + config ───────────────────────────────────────────────────
import os, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE   = 512
BATCH_SIZE = 8
NUM_EPOCHS = 20
LR         = 3e-4
NUM_CLASSES= 10

CLASS_NAMES = [
    "Background", "Bldg Flooded", "Bldg Non-Flooded",
    "Road Flooded", "Road Non-Flooded", "Water",
    "Tree", "Vehicle", "Pool", "Grass"
]

COLORMAP = np.array([
    [0,0,0],[255,0,0],[0,128,0],[255,165,0],[128,128,128],
    [0,0,255],[34,139,34],[255,255,0],[0,255,255],[144,238,144]
], dtype=np.uint8)

print(f"Config: {IMG_SIZE}px | batch={BATCH_SIZE} | epochs={NUM_EPOCHS} | lr={LR}")

In [ ]:
# ── Cell 4: Dataset + Augmentations ───────────────────────────────────────────
# ImageNet mean/std — matches pretrained ResNet50 encoder
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


class FloodNetDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir   = img_dir
        self.mask_dir  = mask_dir
        self.img_files = sorted(os.listdir(img_dir))
        self.transform = transform

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_fname  = self.img_files[idx]
        mask_fname = img_fname.replace(".jpg", "_lab.png")

        img  = np.array(Image.open(os.path.join(self.img_dir,  img_fname)).convert("RGB"))
        mask = np.array(Image.open(os.path.join(self.mask_dir, mask_fname)))

        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out["image"]
            mask = out["mask"]

        return img, mask.long()

    def has_flood(self, idx):
        """Returns True if this mask contains flooded buildings (class 1)."""
        mask_fname = self.img_files[idx].replace(".jpg", "_lab.png")
        mask = np.array(Image.open(os.path.join(self.mask_dir, mask_fname)))
        return bool((mask == 1).any())


# ── Build datasets ────────────────────────────────────────────────────────────
train_ds = FloodNetDataset(TRAIN_IMG, TRAIN_MASK, train_transform)
val_ds   = FloodNetDataset(VAL_IMG,   VAL_MASK,   val_transform)

# ── DATA FIX 1: Weighted sampler — oversample flood images 3× ─────────────────
# EDA showed only 10.3% of images have flooded buildings.
# Without this, the model rarely sees the most important class.
print("Building weighted sampler (scanning train masks)...")
sample_weights = []
for idx in range(len(train_ds)):
    sample_weights.append(3.0 if train_ds.has_flood(idx) else 1.0)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_ds), replacement=True)

flood_count = sum(1 for w in sample_weights if w > 1)
print(f"Flood images: {flood_count}/1445 ({flood_count/1445*100:.1f}%) — weighted 3× in sampling")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# ── Cell 5: Model + Loss + Optimiser ──────────────────────────────────────────

# ── Model: U-Net with ResNet50 encoder (pretrained ImageNet) ──────────────────
# ResNet50 chosen over EfficientNet-B4 for speed on T4 (same quality, faster)
model = smp.Unet(
    encoder_name    = "resnet50",
    encoder_weights = "imagenet",
    in_channels     = 3,
    classes         = NUM_CLASSES,
).to(DEVICE)

# ── DATA FIX 2: Class weights — inverse frequency, background = 0 ──────────────
# EDA pixel counts (approximate from scan): set background weight=0
# so the model ignores unannotated image edges (class 0 is an artifact).
# Rare classes (Vehicle, Pool) get high weights so loss penalises missing them.
raw_weights = torch.tensor([
    0.0,   # 0 Background  ← excluded from loss (image edge artifact)
    4.0,   # 1 Bldg Flooded  ← most important
    2.5,   # 2 Bldg Non-Flooded
    3.5,   # 3 Road Flooded
    1.5,   # 4 Road Non-Flooded
    2.0,   # 5 Water
    0.5,   # 6 Tree
    4.5,   # 7 Vehicle       ← very rare
    4.5,   # 8 Pool          ← very rare
    0.3,   # 9 Grass         ← dominant, penalise less
], dtype=torch.float32).to(DEVICE)

# ── Combined loss: DiceLoss + weighted CrossEntropyLoss ───────────────────────
dice_loss = smp.losses.DiceLoss(mode='multiclass', ignore_index=0)
ce_loss   = nn.CrossEntropyLoss(weight=raw_weights, ignore_index=0)

def combined_loss(pred, target):
    return dice_loss(pred, target) + ce_loss(pred, target)

# ── Optimiser + scheduler ────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1,
)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model: U-Net + ResNet50 — {total_params:.1f}M parameters")
print(f"Loss: DiceLoss + CrossEntropyLoss (background ignored, flood weighted 4×)")
print(f"Optimiser: AdamW lr={LR} | Scheduler: OneCycleLR")

In [ ]:
# ── Cell 6: Metrics helper ────────────────────────────────────────────────────
def compute_iou_per_class(preds, targets, num_classes=10):
    """Compute IoU for each class. Returns array of shape (num_classes,)."""
    iou = np.zeros(num_classes)
    for c in range(1, num_classes):  # skip background (class 0)
        pred_c   = (preds == c)
        target_c = (targets == c)
        intersection = (pred_c & target_c).sum()
        union        = (pred_c | target_c).sum()
        iou[c] = intersection / (union + 1e-6) if union > 0 else float('nan')
    return iou


def evaluate(model, loader, device):
    """Run validation, return mean loss and per-class IoU."""
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            loss   = combined_loss(logits, masks)
            total_loss += loss.item()

            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(masks.cpu().numpy())

    all_preds   = np.concatenate([p.flatten() for p in all_preds])
    all_targets = np.concatenate([t.flatten() for t in all_targets])

    iou_per_class = compute_iou_per_class(all_preds, all_targets)
    miou = np.nanmean(iou_per_class[1:])  # exclude background

    return total_loss / len(loader), iou_per_class, miou

print("Metrics helpers ready.")

In [ ]:
# ── Cell 7: Training loop ─────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "miou": [], "flood_iou": []}
best_miou   = 0.0
best_ckpt   = os.path.join(CKPT_DIR, "best_model.pth")

print(f"Starting training — {NUM_EPOCHS} epochs on {DEVICE}")
print(f"Estimated time: ~{NUM_EPOCHS * 2}-{NUM_EPOCHS * 3} minutes on T4")
print("-" * 72)
print(f"{'Ep':>3} {'TrainLoss':>10} {'ValLoss':>9} {'mIoU':>7} {'FloodIoU':>10} {'Time':>7}")
print("-" * 72)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = combined_loss(logits, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Validate ─────────────────────────────────────────────────────────────
    val_loss, iou_per_class, miou = evaluate(model, val_loader, DEVICE)
    flood_iou = iou_per_class[1]  # class 1 = Bldg Flooded

    elapsed = time.time() - t0
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["miou"].append(miou)
    history["flood_iou"].append(float(flood_iou))

    print(f"{epoch:>3} {train_loss:>10.4f} {val_loss:>9.4f} {miou:>7.4f} {float(flood_iou):>10.4f} {elapsed:>5.0f}s")

    # ── Save best ─────────────────────────────────────────────────────────────
    if miou > best_miou:
        best_miou = miou
        torch.save({
            "epoch":     epoch,
            "model":     model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "miou":      miou,
            "flood_iou": float(flood_iou),
        }, best_ckpt)
        print(f"   ✓ Saved best model (mIoU={miou:.4f})")

print("-" * 72)
print(f"Training complete. Best mIoU = {best_miou:.4f}")
print(f"Checkpoint saved to: {best_ckpt}")

In [ ]:
# ── Cell 8: Training curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["train_loss"], label="Train", color="royalblue")
axes[0].plot(history["val_loss"],   label="Val",   color="tomato")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history["miou"], color="green")
axes[1].set_title("Validation mIoU (excl. background)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mIoU")
axes[1].grid(alpha=0.3)

axes[2].plot(history["flood_iou"], color="red")
axes[2].set_title("IoU — Bldg Flooded (class 1) ← key metric")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("IoU")
axes[2].grid(alpha=0.3)

plt.suptitle(f"FloodNet U-Net Training — Best mIoU: {best_miou:.4f}", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "training_curves.png"), dpi=150)
plt.show()

In [ ]:
# ── Cell 9: Per-class IoU on validation set ───────────────────────────────────
# Load best checkpoint
ckpt = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
print(f"Loaded best model from epoch {ckpt['epoch']} (mIoU={ckpt['miou']:.4f})")
print()

_, iou_per_class, miou = evaluate(model, val_loader, DEVICE)

print(f"{'ID':<4} {'Class':<22} {'IoU':>8}  {'Bar'}")
print("-" * 50)
for i in range(1, 10):  # skip background
    iou_val = iou_per_class[i]
    bar = '█' * int(iou_val * 30) if not np.isnan(iou_val) else ''
    flag = " ← KEY" if i == 1 else ""
    print(f"{i:<4} {CLASS_NAMES[i]:<22} {iou_val:>8.4f}  {bar}{flag}")
print("-" * 50)
print(f"     {'mIoU (excl bg)':<22} {miou:>8.4f}")
print()
print(f"Baseline to beat: mIoU ~0.03 (predict all Grass)")
print(f"Our model mIoU  : {miou:.4f}")

In [ ]:
# ── Cell 10: Visual predictions on test images ────────────────────────────────
def mask_to_rgb(mask_arr):
    return COLORMAP[mask_arr]

model.eval()
test_ds     = FloodNetDataset(TEST_IMG, TEST_MASK, val_transform)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

# Find 4 test images with flooded buildings
test_imgs_list = sorted(os.listdir(TEST_IMG))
flood_test_idxs = []
for idx, fname in enumerate(test_imgs_list):
    mask_fname = fname.replace(".jpg", "_lab.png")
    mask_arr = np.array(Image.open(os.path.join(TEST_MASK, mask_fname)))
    if (mask_arr == 1).any():
        flood_test_idxs.append(idx)
    if len(flood_test_idxs) == 4:
        break

n_show = len(flood_test_idxs)
fig, axes = plt.subplots(n_show, 3, figsize=(15, n_show * 4))

IMAGENET_MEAN_T = torch.tensor(IMAGENET_MEAN).view(3,1,1)
IMAGENET_STD_T  = torch.tensor(IMAGENET_STD).view(3,1,1)

for row, idx in enumerate(flood_test_idxs):
    img_tensor, mask_tensor = test_ds[idx]

    with torch.no_grad():
        logits = model(img_tensor.unsqueeze(0).to(DEVICE))
        pred   = logits.argmax(dim=1).squeeze(0).cpu().numpy()

    mask_np = mask_tensor.numpy()

    # Denormalise image for display
    img_display = (img_tensor * IMAGENET_STD_T + IMAGENET_MEAN_T)
    img_display = img_display.permute(1,2,0).numpy().clip(0,1)

    pred_rgb = mask_to_rgb(pred)
    gt_rgb   = mask_to_rgb(mask_np)

    flood_iou_img = float(compute_iou_per_class(pred, mask_np)[1])

    axes[row,0].imshow(img_display); axes[row,0].set_title(f"Photo"); axes[row,0].axis("off")
    axes[row,1].imshow(gt_rgb);      axes[row,1].set_title("Ground Truth"); axes[row,1].axis("off")
    axes[row,2].imshow(pred_rgb);    axes[row,2].set_title(f"Predicted  (flood IoU={flood_iou_img:.3f})"); axes[row,2].axis("off")

legend_patches = [
    mpatches.Patch(color=tuple(c/255 for c in COLORMAP[i]), label=f"{i}: {CLASS_NAMES[i]}")
    for i in range(1, 10)
]
fig.legend(handles=legend_patches, loc='lower center', ncol=5, fontsize=9, bbox_to_anchor=(0.5,-0.01))
plt.suptitle("Predictions on test images containing flooded buildings", fontsize=13, y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "test_predictions.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Predictions saved to Drive.")